In [1]:
# Core imports
import os, sys, asyncio, yaml
from dotenv import load_dotenv
from pydantic import BaseModel
from contextlib import asynccontextmanager
# Project modules (assumes this notebook lives in src/mcp_client/)
from client import MCPClient
import json
import requests

In [2]:
import logging
import sys
from typing import Optional

DEFAULT_FORMAT = "%(asctime)s - %(name)s - %(levelname)s - %(message)s"

def get_logger(
    name: str,
    log_file: Optional[str] = None,
    level: int = logging.INFO,
    console_level: Optional[int] = None,
    fmt: str = DEFAULT_FORMAT,
    propagate: bool = False,
):
    """Return a configured logger.

    Ensures we don't attach duplicate handlers if called multiple times.
    """
    logger = logging.getLogger(name)
    logger.setLevel(level)
    logger.propagate = propagate

    formatter = logging.Formatter(fmt)

    # Add/ensure file handler
    if log_file:
        if not any(isinstance(h, logging.FileHandler) and getattr(h, 'baseFilename', None) and h.baseFilename.endswith(log_file) for h in logger.handlers):
            fh = logging.FileHandler(log_file)
            fh.setLevel(level)
            fh.setFormatter(formatter)
            logger.addHandler(fh)

    # Add/ensure console handler
    if console_level is None:
        console_level = level
    if not any(isinstance(h, logging.StreamHandler) and getattr(h, 'stream', None) is sys.stdout for h in logger.handlers):
        ch = logging.StreamHandler(sys.stdout)
        ch.setLevel(console_level)
        ch.setFormatter(formatter)
        logger.addHandler(ch)

    return logger

In [3]:
load_dotenv()
api_key = os.getenv("OPENROUTER_API_KEY", "")

logger = get_logger("mcp-client", log_file="mcp_client.log", level=20, console_level=20)

from pathlib import Path
file_path = Path("/home/vmadmin/intent/src/test/teste.csv")

In [4]:
# Create a client, connect, and process an intent (returns an OpenAI Responses API object)
client = MCPClient(logger=logger, rapp="http://10.233.7.113:8090", file_path= file_path)
connected = await client.connect_to_server("http://127.0.0.1:8000/sse")
await client.set_llm(api_key=api_key, llm_name="qwen",llm_model="qwen/qwen3-235b-a22b")
if not connected:
    raise RuntimeError("Failed to connect to MCP server")

# # 'response' is an object (not a dict) with an 'output' attribute (a list of bl1ocks)
response = await client.process_intent(
   "Create a slice to control autonomous delivery robots in an industrial zone, requiring packet delay budget of 3 ms and uplink rates of 20 Mbps per device."
)

2026-09-20 22:53:02,634 - mcp-client - INFO - Attempting to connect to server at http://127.0.0.1:8000/sse.
2026-09-20 22:53:02,817 - mcp-client - INFO - Requesting MCP tools from the server.
2026-09-20 22:53:02,824 - mcp-client - INFO - Successfully connected to server. Available tools: ['create_session', 'get_session', 'delete_session', 'ping_api']
2026-09-20 22:53:02,825 - mcp-client - INFO - Setting llm qwen - model qwen/qwen3-235b-a22b
2026-09-20 22:53:02,826 - mcp-client - INFO - Requesting MCP tools from the server.
2026-09-20 22:53:02,849 - mcp-client - INFO - LLM configuration completed successfully.
2026-09-20 22:53:02,850 - mcp-client - INFO - Calling LLM: qwen
2026-09-20 22:53:33,213 - mcp-client - INFO - Assistant response: ChatCompletion(id='gen-1789959183-zrNQ5hwvlfaTWfNiM3R1', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='I need a few more details to complete the slice creation:\n\n1. Maximum number of devices (UEs)

In [5]:
print(response)

{'role': 'assistant', 'content': 'I need a few more details to complete the slice creation:\n\n1. Maximum number of devices (UEs) expected in this deployment?\n2. Required downstream rate per device (e.g., "20 Mbps")?\n3. Downstream delay budget (already have 3ms uplink specified)?\n\nThe industrial zone service area and service time can be left unspecified if not required.'}
